# Overview of Data Engineering in Applied Data Science



## 1. Introduction

Applied Data Science applies data science techniques (statistics, ML, visualization) to real-world problems, often in production settings like recommendation systems, fraud detection, or predictive maintenance.

Data Engineering is the foundational layer: it builds scalable, reliable systems to **collect, store, clean, and deliver** high-quality data to data scientists and applications.

Without strong data engineering, even the best models fail due to bad, missing, or delayed data.

## 2. What is Data Engineering?

Data Engineers create and maintain the **data infrastructure** and **pipelines**.

Key responsibilities:
- Ingesting data from diverse sources (databases, APIs, logs, IoT)
- Transforming raw data into usable formats (cleaning, joining, aggregating)
- Storing data efficiently (lakes, warehouses, lakehouses)
- Orchestrating workflows and ensuring reliability/scalability
- Monitoring data quality, lineage, and governance

In applied data science teams, data engineers enable data scientists to focus on insights rather than data wrangling.

## 3. Data Engineering vs. Data Science vs. Data Analytics

| Role              | Primary Focus                  | Key Tools/Skills                     | Output                          |
|-------------------|--------------------------------|--------------------------------------|---------------------------------|
| Data Engineering | Pipelines & Infrastructure     | SQL, Spark, Airflow, Kafka, Cloud    | Reliable, scalable data platforms |
| Data Science     | Modeling & Experimentation     | Python/R, Scikit-learn, TensorFlow, Stats | Predictive models, experiments  |
| Data Analytics   | Reporting & Business Insights  | SQL, Tableau/Power BI, Excel         | Dashboards, KPIs, reports       |

In practice, roles overlap in small teams, but clear separation maximizes efficiency in large applied data science projects.

## 4. ETL vs ELT and Data Pipelines

**ETL** (Extract-Transform-Load): Transform data *before* loading (traditional, good for compliance-heavy environments).

**ELT** (Extract-Load-Transform): Load raw data first, transform inside the warehouse (modern, leverages cloud compute).

Data Pipelines automate these steps as DAGs (Directed Acyclic Graphs) for repeatability.

Below is a simple end-to-end pipeline example in pure Python.

In [1]:
import pandas as pd
import numpy as np

# 1. EXTRACT: Simulate pulling data from source (e.g., API or DB)
def extract_data():
    raw_data = {
        'customer_id': [101, 102, 103, 104, 105],
        'purchase_amount': [250.75, None, 175.50, 300.00, 89.99],
        'purchase_date': ['2026-02-01', '2026-02-02', '2026-02-03', '2026-02-04', '2026-02-05'],
        'category': ['Electronics', 'Books', 'Clothing', 'Electronics', 'Books']
    }
    return pd.DataFrame(raw_data)

# 2. TRANSFORM: Clean, enrich, feature engineering
def transform_data(df):
    df = df.copy()
    df['purchase_date'] = pd.to_datetime(df['purchase_date'])
    # Fill missing values with median
    df['purchase_amount'] = df['purchase_amount'].fillna(df['purchase_amount'].median())
    # Create categories
    df['amount_category'] = pd.cut(df['purchase_amount'], 
                                   bins=[0, 150, 300, float('inf')], 
                                   labels=['Low', 'Medium', 'High'])
    df['month'] = df['purchase_date'].dt.month
    return df

# 3. LOAD: Persist to storage (CSV for demo; in prod: database, S3, etc.)
def load_data(df, filename='processed_purchases.csv'):
    df.to_csv(filename, index=False)
    print(f'Data successfully loaded to {filename}')
    return df

# Run full pipeline
print('=== Starting ETL Pipeline ===')
raw_df = extract_data()
print('\nRaw Data (Extracted):')
print(raw_df)

transformed_df = transform_data(raw_df)
print('\nTransformed Data:')
print(transformed_df)

final_df = load_data(transformed_df)
print('\n=== Pipeline Completed Successfully! ===')
print('\nFinal shape:', final_df.shape)

=== Starting ETL Pipeline ===

Raw Data (Extracted):
   customer_id  purchase_amount purchase_date     category
0          101           250.75    2026-02-01  Electronics
1          102              NaN    2026-02-02        Books
2          103           175.50    2026-02-03     Clothing
3          104           300.00    2026-02-04  Electronics
4          105            89.99    2026-02-05        Books

Transformed Data:
   customer_id  purchase_amount purchase_date     category amount_category  \
0          101          250.750    2026-02-01  Electronics          Medium   
1          102          213.125    2026-02-02        Books          Medium   
2          103          175.500    2026-02-03     Clothing          Medium   
3          104          300.000    2026-02-04  Electronics          Medium   
4          105           89.990    2026-02-05        Books             Low   

   month  
0      2  
1      2  
2      2  
3      2  
4      2  
Data successfully loaded to processed_p

## 5. Data Storage Architectures

- **Data Lake**: Cheap storage for raw data (S3, GCS + formats like Parquet, ORC).
- **Data Warehouse**: Structured, fast SQL queries (Snowflake, BigQuery, Redshift).
- **Data Lakehouse**: Combines both with ACID, schema enforcement (Delta Lake, Iceberg, Hudi).

In applied data science, lakehouses are increasingly popular for both analytics and ML feature serving.

## 6. Orchestration and Modern Tools (2026 Landscape)

**Orchestration** (scheduling, dependency management, retries):
- Apache Airflow / Astronomer
- Prefect / Dagster / Mage

**Processing**:
- Batch: Apache Spark, Polars, DuckDB
- Streaming: Apache Kafka + Flink / ksqlDB

**Transformation**: dbt (data build tool)

**Cloud-native**: AWS Glue/Data Pipeline, GCP Dataflow/Composer, Azure Data Factory/Synapse

**Data Quality & Observability**: Great Expectations, Monte Carlo, Elementary

**MLOps intersection**: Feature stores (Feast, Tecton), model monitoring.

## 7. Best Practices in Applied Data Engineering

- **Idempotency & Reproducibility**: Pipelines should be rerun safely.
- **Data Lineage & Catalog**: Track where data comes from (Amundsen, DataHub).
- **Testing**: Unit tests for transformations, schema validation.
- **Monitoring**: Alerts on failures, data volume anomalies.
- **CI/CD**: Treat pipelines as code (GitHub Actions, GitLab CI).
- **Security & Compliance**: Encryption, access control, GDPR/HIPAA.

Adopt Infrastructure-as-Code (Terraform) and GitOps.